In [5]:
import sys
import os
import math
sys.path.append("../")
sys.path.append("../../")

In [6]:
!export LD_LIBRARY_PATH=$LD_LIBRARY_PATH:/home/dongyoonhwang/.mujoco/mujoco210/bin:/usr/lib/nvidia
!export MUJOCO_PY_MJKEY_PATH=/home/nas2_userI/byungkunlee/.mujoco/mjkey.txt
!export MJKEY_PATH=/home/nas2_userI/byungkunlee/.mujoco/mjkey.txt
!export MJLIB_PATH=/home/nas2_userI/byungkunlee/.mujoco/mujoco210/bin/libmujoco210.so
!export MUJOCO_PY_MUJOCO_PATH=/home/dongyoonhwang/.mujoco/mujoco210

# >>> MuJoCo visualization >>>
!export XLA_PYTHON_CLIENT_PREALLOCATE=false
!export MUJOCO_GL='egl'
!export MUJOCO_EGL_DEVICE_ID='0'
!export MKL_SERVICE_FORCE_INTEL='0'

In [7]:
from scale_rl.common.wandb_utils import *

In [8]:
MUJ_STEPS = 1000000 # 1M
DMC_STEPS = 1000000 # 1M
MYO_STEPS = 1000000 # 1M
HB_STEPS = 1000000 # 1M

In [9]:
from scale_rl.envs.mujoco import MUJOCO_ALL, MUJOCO_RANDOM_SCORE, MUJOCO_TD3_SCORE
from scale_rl.envs.dmc import DMC_EASY_MEDIUM, DMC_HARD
from scale_rl.envs.humanoid_bench import HB_LOCOMOTION_NOHAND, HB_RANDOM_SCORE, HB_SUCCESS_SCORE
from scale_rl.envs.myosuite import MYOSUITE_TASKS

def replace_hypen_to_underbar(env_name_list):
    for idx in range(len(env_name_list)):
        env_name_list[idx] = env_name_list[idx].replace('_', '-')
    return env_name_list
def replace_hypen_to_underbar_dict(env_name_dict):
    _new_dict = {}
    for k, v in env_name_dict.items():
        _new_dict[k.replace('_', '-')] = v
    return _new_dict

MUJOCO_ALL = replace_hypen_to_underbar(MUJOCO_ALL)
DMC_EASY_MEDIUM = replace_hypen_to_underbar(DMC_EASY_MEDIUM)
DMC_HARD = replace_hypen_to_underbar(DMC_HARD)
HB_LOCOMOTION_NOHAND = replace_hypen_to_underbar(HB_LOCOMOTION_NOHAND)
MYOSUITE_TASKS = replace_hypen_to_underbar(MYOSUITE_TASKS)
HB_LOCOMOTION_NOHAND = replace_hypen_to_underbar(HB_LOCOMOTION_NOHAND)

MUJOCO_RANDOM_SCORE = replace_hypen_to_underbar_dict(MUJOCO_RANDOM_SCORE)
MUJOCO_TD3_SCORE = replace_hypen_to_underbar_dict(MUJOCO_TD3_SCORE)

HB_RANDOM_SCORE = replace_hypen_to_underbar_dict(HB_RANDOM_SCORE)
HB_SUCCESS_SCORE = replace_hypen_to_underbar_dict(HB_SUCCESS_SCORE)

DMC_ALL = [*DMC_EASY_MEDIUM, *DMC_HARD]

In [ ]:
eval_df = pd.read_csv('/home/nas4_user/youngdolee/SimbaV2_fix/results/baseline/redq_utd20.csv', index_col=0)
eval_df['env_name'] = eval_df['env_name'].str.replace('_', '-')

In [11]:
overall_df = []
for env_type in ['MUJOCO_ALL', 'DMC_EASY_MEDIUM', 'DMC_HARD', 'MYOSUITE', 'HB_LOCOMOTION_NOHAND']:
    metric_type = 'avg_return'
    if env_type == 'MUJOCO_ALL':
        env_list = MUJOCO_ALL
        env_step = MUJ_STEPS
        
    elif env_type == 'DMC_EASY_MEDIUM':
        env_list = DMC_EASY_MEDIUM
        env_step = DMC_STEPS

    elif env_type == 'DMC_HARD':
        env_list = DMC_HARD
        env_step = DMC_STEPS

    elif env_type == 'MYOSUITE':
        env_list = MYOSUITE_TASKS
        env_step = MYO_STEPS
        metric_type = 'avg_success'

    elif env_type == 'HB_LOCOMOTION_NOHAND':
        env_list = HB_LOCOMOTION_NOHAND
        env_step = HB_STEPS
        
    _eval_df = eval_df[eval_df['env_name'].isin(env_list)]
    if len(_eval_df) == 0 or env_type == 'DMC_EASY_MEDIUM':
        continue
    _eval_df = _eval_df[_eval_df["env_step"] <= env_step]
    _eval_df = _eval_df[_eval_df["metric"].isin([metric_type])]
    assert max(_eval_df["env_step"]) == env_step
    _eval_df = _eval_df[_eval_df["env_step"] == env_step]
    num_seeds = _eval_df["seed"].nunique()

    if env_type == 'MUJOCO_ALL':
        _eval_df = normalize_score_with_random_and_base_score(_eval_df, MUJOCO_RANDOM_SCORE, MUJOCO_TD3_SCORE)

    if env_type == 'HB_LOCOMOTION_NOHAND':
        _eval_df = normalize_score_with_random_and_base_score(_eval_df, HB_RANDOM_SCORE, HB_SUCCESS_SCORE)

    # to match the scale of success to the return in comparison
    if env_type in ['DMC_EASY_MEDIUM', 'DMC_HARD']:
        _eval_df.loc[:, 'value'] /= 1000.0
        
    overall_df.append(_eval_df)
        
    grouped_data = _eval_df.groupby("env_step")["value"]
    mean = float(grouped_data.mean().values)
    std_err = float(grouped_data.sem().values)
    
    print(f"{env_type} - number of seeds: {num_seeds}")
    print(f"{round(mean, 3)} ± {round(1.96 * std_err, 3)}")
    
overall_df = pd.concat(overall_df)
mean = overall_df["value"].mean()
std_err = overall_df["value"].sem()
print(f"Overall: {round(mean, 3)} ± {round(1.96 * std_err, 3)}")

MUJOCO_ALL - number of seeds: 10
1.465 ± 0.127
DMC_HARD - number of seeds: 10
0.723 ± 0.061
Overall: 1.032 ± 0.091
